# Benchmark Forecasting — Coûts GCP Veolia

Ce notebook prend en entrée les fichiers exportés par `Timeseries.ipynb` et compare **7 approches** de prévision, dont 2 **Event-Aware** intégrant les métriques Cloud Run.

## Modèles Baseline

| # | Modèle | Famille | Librairie |
|---|--------|---------|-----------|
| 1 | **AutoARIMA** | Statistique classique | Nixtla `statsforecast` |
| 2 | **AutoETS** | Lissage exponentiel | Nixtla `statsforecast` |
| 3 | **AutoTheta** | Méthode Theta | Nixtla `statsforecast` |
| 4 | **TimesNet** | Deep Learning 2D (FFT + TimesBlock) | Nixtla `neuralforecast` |
| 5 | **N-HiTS** | Interpolation hiérarchique neurale | Nixtla `neuralforecast` |
| 6 | **Prophet** | Additif décomposable | Meta |

## Modèles Event-Aware (covariables Cloud Run)

| # | Modèle | Signal injecté |
|---|--------|----------------|
| 7 | **TimesNet + Events** | CPU, Instances, Requests, Latency → `hist_exog_list` |
| 8 | **Prophet + Events** | CPU, Instances, Requests, Latency → `add_regressor()` |

## Pipeline Event-Aware

```
Cloud Run CSVs (6h)
    → agrégation journalière
    → normalisation min-max
    → corrélation croisée (lag optimal)
    → FFT (périodes dominantes → top_k TimesBlock)
    → injection dans TimesNet (hist_exog) & Prophet (regressor)
    → benchmark MAE / RMSE / MAPE / R² : Baseline vs Event-Aware
```

**Métriques** : MAE, RMSE, MAPE (%), R² — classement par score composite (rang moyen)

> **Prérequis** : lancer la cellule `Export pour Benchmark_Forecasting.ipynb` de `Timeseries.ipynb`.

In [19]:
# Installation des dépendances (à lancer une seule fois)
# !pip install statsforecast neuralforecast prophet scikit-learn plotly pandas pyarrow scipy

In [20]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Imports de base OK')

Imports de base OK


## 0. Chargement des données

In [21]:
# Chargement depuis les parquets exportés par Timeseries.ipynb
daily = pd.read_parquet('daily_costs.parquet')
daily['ds'] = pd.to_datetime(daily['ds'])
daily = daily.sort_values('ds').reset_index(drop=True)

per_service = pd.read_parquet('daily_per_service.parquet')
per_service['ds'] = pd.to_datetime(per_service['ds'])
per_service = per_service.sort_values('ds').reset_index(drop=True)

print(f'  Série journalière : {len(daily)} jours  ({daily["ds"].min().date()} → {daily["ds"].max().date()})')
print(f'  Services disponibles : {len(per_service.columns) - 1}')
daily.head()

  Série journalière : 170 jours  (2026-01-05 → 2026-06-23)
  Services disponibles : 9


,ds,y
0,2026-01-05,0.0
1,2026-01-06,0.0
2,2026-01-07,0.0
3,2026-01-08,0.0
4,2026-01-09,0.0


In [22]:
# ── Split train / test ────────────────────────────────────────────────────────
HORIZON = 30          # jours à prédire
n_total  = len(daily)
n_test   = HORIZON
n_train  = n_total - n_test

train = daily.iloc[:n_train].copy()
test  = daily.iloc[n_train:].copy()

print(f'  Train : {n_train} jours  ({train["ds"].min().date()} → {train["ds"].max().date()})')
print(f'  Test  : {n_test} jours   ({test["ds"].min().date()}  → {test["ds"].max().date()})')

# Visualisation du split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train',
                         line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines', name='Test (ground truth)',
                         line=dict(color='orange')))
fig.add_vrect(x0=test['ds'].min(), x1=test['ds'].max(),
              fillcolor='rgba(255,165,0,0.08)', line_width=0,
              annotation_text=f'Zone test ({n_test}j)', annotation_position='top left')
fig.update_layout(title='Split Train / Test — Coût journalier total',
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

  Train : 140 jours  (2026-01-05 → 2026-05-24)
  Test  : 30 jours   (2026-05-25  → 2026-06-23)


In [23]:
# ── Métriques ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, model_name):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    r2   = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'MAPE (%)': round(mape, 2), 'R²': round(r2, 4)}

results = []   # accumulateur des métriques
forecasts_all = {}   # stockage des prévisions pour la viz finale
y_test = test['y'].values
dates_test = test['ds'].values

print('  Métriques initialisées — prêt pour les modèles.')

  Métriques initialisées — prêt pour les modèles.


---
## Modèle 1 — AutoARIMA  *(Nixtla StatsForecast)*

AutoARIMA sélectionne automatiquement les ordres `(p,d,q)(P,D,Q)s` par minimisation de l'AIC. Adapté aux séries avec tendance et/ou saisonnalité hebdomadaire (`season_length=7`).

In [24]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

# Format Nixtla : unique_id, ds, y
train_sf = train.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

sf_arima = StatsForecast(
    models=[AutoARIMA(season_length=7, approximation=True)],
    freq='D',
    n_jobs=-1,
    verbose=False
)
sf_arima.fit(train_sf)
fc_arima = sf_arima.predict(h=HORIZON)

y_pred_arima = fc_arima['AutoARIMA'].values
metrics_arima = compute_metrics(y_test, y_pred_arima, 'AutoARIMA')
results.append(metrics_arima)
forecasts_all['AutoARIMA'] = y_pred_arima

print(f"  AutoARIMA — MAE={metrics_arima['MAE']:.2f}€  RMSE={metrics_arima['RMSE']:.2f}€  "
      f"MAPE={metrics_arima['MAPE (%)']:.1f}%  R²={metrics_arima['R²']:.4f}")

# Graphique
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_arima, mode='lines', name='AutoARIMA',
                         line=dict(color='crimson', dash='dash')))
fig.update_layout(title=f"AutoARIMA | R²={metrics_arima['R²']:.4f} | MAE={metrics_arima['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

  AutoARIMA — MAE=6.94€  RMSE=9.20€  MAPE=27.8%  R²=-0.2478


---
## Modèle 2 — AutoETS + AutoTheta  *(Nixtla StatsForecast)*

- **AutoETS** : sélection automatique des composantes Erreur/Tendance/Saisonnalité.
- **AutoTheta** : variante de la méthode Theta, efficace sur séries courtes.

In [25]:
from statsforecast.models import AutoETS, AutoTheta

sf_ets = StatsForecast(
    models=[AutoETS(season_length=7), AutoTheta(season_length=7)],
    freq='D',
    n_jobs=-1,
    verbose=False
)
sf_ets.fit(train_sf)
fc_ets = sf_ets.predict(h=HORIZON)

y_pred_ets   = fc_ets['AutoETS'].values
y_pred_theta = fc_ets['AutoTheta'].values

for name, pred in [('AutoETS', y_pred_ets), ('AutoTheta', y_pred_theta)]:
    m = compute_metrics(y_test, pred, name)
    results.append(m)
    forecasts_all[name] = pred
    print(f"  {name:<12} — MAE={m['MAE']:.2f}€  RMSE={m['RMSE']:.2f}€  MAPE={m['MAPE (%)']:.1f}%  R²={m['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_ets,   mode='lines', name='AutoETS',   line=dict(color='green',  dash='dash')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_theta, mode='lines', name='AutoTheta', line=dict(color='purple', dash='dot')))
fig.update_layout(title='AutoETS vs AutoTheta', xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

  AutoETS      — MAE=5.65€  RMSE=8.03€  MAPE=23.9%  R²=0.0495
  AutoTheta    — MAE=5.66€  RMSE=7.88€  MAPE=24.8%  R²=0.0849


---
## Modèle 3 — TimesNet  *(Nixtla NeuralForecast)*

TimesNet transforme la série 1D en représentation **2D** pour capturer les variations intra-période et inter-période via des blocs `TimesBlock` (convolutions 2D). Architecture proposée par Wu et al. (2023) — *TimesNet: Temporal 2D-Variation Modeling for General Time Series Analysis*.

In [28]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TimesNet

# Format NeuralForecast : unique_id, ds, y
train_nf = train.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

INPUT_SIZE = 2 * HORIZON  # fenêtre de contexte = 2× l'horizon

timesnet_model = TimesNet(
    h=HORIZON,
    input_size=INPUT_SIZE,
    encoder_layers=2,
    dropout=0.1,
    top_k=3,
    num_kernels=4,
    max_steps=300,
    batch_size=8,
    learning_rate=1e-3,
    random_seed=42,
    scaler_type='standard',
)

nf_timesnet = NeuralForecast(models=[timesnet_model], freq='D')
nf_timesnet.fit(train_nf)
fc_timesnet = nf_timesnet.predict()

y_pred_timesnet = fc_timesnet['TimesNet'].values
metrics_timesnet = compute_metrics(y_test, y_pred_timesnet, 'TimesNet')
results.append(metrics_timesnet)
forecasts_all['TimesNet'] = y_pred_timesnet

print(f"  TimesNet — MAE={metrics_timesnet['MAE']:.2f}€  RMSE={metrics_timesnet['RMSE']:.2f}€  "
      f"MAPE={metrics_timesnet['MAPE (%)']:.1f}%  R²={metrics_timesnet['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_timesnet, mode='lines', name='TimesNet',
                         line=dict(color='darkred', dash='dash')))
fig.update_layout(title=f"TimesNet | R²={metrics_timesnet['R²']:.4f} | MAE={metrics_timesnet['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name           | Type          | Params | Mode 
---------------------------------------------------------
0 | loss           | MAE           | 0      | train
1 | padder_train   | ConstantPad1d | 0      | train
2 | scaler         | TemporalNorm  | 0      | train
3 | model          | ModuleList    | 1.4 M  | train
4 | enc_embedding  | DataEmbedding | 192    | train
5 | layer_norm     | LayerNorm     | 128    | train
6 | predict_linear | Linear        | 5.5 K  | train
7 | projection     | Linear        | 65     | train
---------------------------------------------------------
1.4 M     Trainable params
0         Non-trainable params
1.4 M     Total params
5.533     Total estimated model params size (MB)
42        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

  TimesNet — MAE=6.46€  RMSE=8.81€  MAPE=29.6%  R²=-0.1453


---
## Modèle 4 — N-HiTS  *(Nixtla NeuralForecast)*

**N-HiTS** (Neural Hierarchical Interpolation for Time Series) décompose la prévision en plusieurs échelles temporelles grâce à du *hierarchical interpolation*, ce qui lui permet d'être très efficace sur des séries longues ou multi-saisonnières. Challu et al. (2023).

In [29]:
from neuralforecast.models import NHITS

nhits_model = NHITS(
    h=HORIZON,
    input_size=2 * HORIZON,
    stack_types=['identity', 'identity', 'identity'],
    n_blocks=[1, 1, 1],
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    interpolation_mode='linear',
    dropout_prob_theta=0.1,
    max_steps=300,
    batch_size=8,
    learning_rate=1e-3,
    random_seed=42,
    scaler_type='standard',
)

nf_nhits = NeuralForecast(models=[nhits_model], freq='D')
nf_nhits.fit(train_nf)
fc_nhits = nf_nhits.predict()

y_pred_nhits = fc_nhits['NHITS'].values
metrics_nhits = compute_metrics(y_test, y_pred_nhits, 'N-HiTS')
results.append(metrics_nhits)
forecasts_all['N-HiTS'] = y_pred_nhits

print(f"  N-HiTS — MAE={metrics_nhits['MAE']:.2f}€  RMSE={metrics_nhits['RMSE']:.2f}€  "
      f"MAPE={metrics_nhits['MAPE (%)']:.1f}%  R²={metrics_nhits['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_nhits, mode='lines', name='N-HiTS',
                         line=dict(color='teal', dash='dash')))
fig.update_layout(title=f"N-HiTS | R²={metrics_nhits['R²']:.4f} | MAE={metrics_nhits['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

Seed set to 42
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores

  | Name         | Type          | Params | Mode 
-------------------------------------------------------
0 | loss         | MAE           | 0      | train
1 | padder_train | ConstantPad1d | 0      | train
2 | scaler       | TemporalNorm  | 0      | train
3 | blocks       | ModuleList    | 683 K  | train
-------------------------------------------------------
683 K     Trainable params
0         Non-trainable params
683 K     Total params
2.733     Total estimated model params size (MB)
43        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=300` reached.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


Predicting: |          | 0/? [00:00<?, ?it/s]

  N-HiTS — MAE=8.08€  RMSE=10.83€  MAPE=35.3%  R²=-0.7304


---
## Modèle 5 — Prophet  *(Meta / awesome-time-series-forecasting)*

**Prophet** modélise la série comme une somme de composantes : tendance + saisonnalité hebdomadaire + saisonnalité annuelle + jours fériés. Robuste aux valeurs manquantes et aux outliers. Figurant en tête des ressources du repo *awesome-time-series-forecasting*.

In [30]:
from prophet import Prophet

train_prophet = train.rename(columns={'ds': 'ds', 'y': 'y'})

# Données insuffisantes pour saisonnalité annuelle (6 mois) → on désactive
prophet_model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.1,
    seasonality_prior_scale=5,
)
prophet_model.fit(train_prophet)

future = prophet_model.make_future_dataframe(periods=HORIZON)
fc_prophet = prophet_model.predict(future)

# Extraire uniquement la zone test
y_pred_prophet = fc_prophet['yhat'].iloc[-HORIZON:].values

metrics_prophet = compute_metrics(y_test, y_pred_prophet, 'Prophet')
results.append(metrics_prophet)
forecasts_all['Prophet'] = y_pred_prophet

print(f"  Prophet — MAE={metrics_prophet['MAE']:.2f}€  RMSE={metrics_prophet['RMSE']:.2f}€  "
      f"MAPE={metrics_prophet['MAPE (%)']:.1f}%  R²={metrics_prophet['R²']:.4f}")

# Graphique avec intervalles de confiance Prophet
prophet_test_fc = fc_prophet.iloc[-HORIZON:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(
    x=pd.concat([pd.Series(prophet_test_fc['ds']), pd.Series(prophet_test_fc['ds'].values[::-1])]),
    y=pd.concat([pd.Series(prophet_test_fc['yhat_upper']), pd.Series(prophet_test_fc['yhat_lower'].values[::-1])]),
    fill='toself', fillcolor='rgba(0,128,0,0.12)', line=dict(color='rgba(255,255,255,0)'),
    name='IC 80%'))
fig.add_trace(go.Scatter(x=prophet_test_fc['ds'], y=prophet_test_fc['yhat'], mode='lines',
                         name='Prophet', line=dict(color='green', dash='dash')))
fig.update_layout(title=f"Prophet | R²={metrics_prophet['R²']:.4f} | MAE={metrics_prophet['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

16:33:35 - cmdstanpy - INFO - Chain [1] start processing
16:33:37 - cmdstanpy - INFO - Chain [1] done processing


  Prophet — MAE=5.75€  RMSE=7.58€  MAPE=26.9%  R²=0.1523


---
## Benchmark — Tableau de performance & Ranking

In [31]:
bench = pd.DataFrame(results)

# Score composite : rang moyen sur MAE, RMSE, MAPE (plus bas = mieux), R² (plus haut = mieux)
bench['Rank_MAE']  = bench['MAE'].rank()
bench['Rank_RMSE'] = bench['RMSE'].rank()
bench['Rank_MAPE'] = bench['MAPE (%)'].rank()
bench['Rank_R2']   = bench['R²'].rank(ascending=False)
bench['Score']     = (bench['Rank_MAE'] + bench['Rank_RMSE'] + bench['Rank_MAPE'] + bench['Rank_R2']) / 4
bench = bench.sort_values('Score').reset_index(drop=True)

display_cols = ['Model', 'MAE', 'RMSE', 'MAPE (%)', 'R²', 'Score']
print('═' * 68)
print('   BENCHMARK — Classement des modèles (Score = rang moyen)')
print('═' * 68)
print(f"  {'#':<3} {'Modèle':<14} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8} {'Score':>7}")
print('-' * 68)
for i, row in bench.iterrows():
    star = ' ★' if i == 0 else ''
    print(f"  {i+1:<3} {row['Model']:<14} {row['MAE']:>7.2f}€ {row['RMSE']:>7.2f}€ "
          f"{row['MAPE (%)']:>7.1f}% {row['R²']:>8.4f} {row['Score']:>7.2f}{star}")
print('═' * 68)
best_model = bench.iloc[0]['Model']
print(f"\n  Meilleur modèle : {best_model}")

bench[display_cols]

════════════════════════════════════════════════════════════════════
   BENCHMARK — Classement des modèles (Score = rang moyen)
════════════════════════════════════════════════════════════════════
  #   Modèle              MAE     RMSE     MAPE       R²   Score
--------------------------------------------------------------------
  1   AutoETS           5.65€    8.03€    23.9%   0.0495    2.00 ★
  2   AutoTheta         5.66€    7.88€    24.8%   0.0849    2.00
  3   Prophet           5.75€    7.58€    26.9%   0.1523    2.00
  4   TimesNet          6.46€    8.81€    29.6%  -0.1453    4.25
  5   AutoARIMA         6.94€    9.20€    27.8%  -0.2478    4.75
  6   N-HiTS            8.08€   10.83€    35.3%  -0.7304    6.00
════════════════════════════════════════════════════════════════════

  Meilleur modèle : AutoETS


,Model,MAE,RMSE,MAPE (%),R²,Score
0,AutoETS,5.65,8.03,23.89,0.0495,2.00
1,AutoTheta,5.66,7.88,24.83,0.0849,2.00
2,Prophet,5.75,7.58,26.89,0.1523,2.00
3,TimesNet,6.46,8.81,29.58,-0.1453,4.25
4,AutoARIMA,6.94,9.20,27.84,-0.2478,4.75
5,N-HiTS,8.08,10.83,35.29,-0.7304,6.00


In [32]:
# ── Graphique métriques comparatives ─────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['MAE (€) — moins = mieux', 'RMSE (€) — moins = mieux',
                    'MAPE (%) — moins = mieux', 'R² — plus = mieux']
)

palette = px.colors.qualitative.Bold
models  = bench['Model'].tolist()
colors  = [palette[i % len(palette)] for i in range(len(models))]

metrics_map = [
    ('MAE',      1, 1, False),
    ('RMSE',     1, 2, False),
    ('MAPE (%)', 2, 1, False),
    ('R²',       2, 2, True),
]
for col, row, col_idx, higher_better in metrics_map:
    vals = bench[col].tolist()
    bar_colors = []
    for i, v in enumerate(vals):
        if higher_better:
            bar_colors.append('gold' if i == 0 else colors[i])
        else:
            bar_colors.append('gold' if i == 0 else colors[i])
    fig.add_trace(
        go.Bar(x=models, y=vals, marker_color=bar_colors,
               text=[f'{v:.2f}' for v in vals], textposition='outside'),
        row=row, col=col_idx
    )

fig.update_layout(title='Comparaison des métriques — tous modèles (or = meilleur)',
                  showlegend=False, height=600)
fig.show()

In [33]:
# ── Graphique comparatif des prévisions vs réel ───────────────────────────────
fig = go.Figure()

# Derniers 30 jours de train pour le contexte visuel
context = train.iloc[-30:]
fig.add_trace(go.Scatter(x=context['ds'], y=context['y'], mode='lines',
                         name='Train (contexte)', line=dict(color='steelblue', width=1.5)))
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines+markers',
                         name='Réel (test)', line=dict(color='black', width=2.5),
                         marker=dict(size=5)))

line_styles = ['dash', 'dot', 'dashdot', 'longdash', 'longdashdot']
model_palette = {'AutoARIMA': 'crimson', 'AutoETS': 'green', 'AutoTheta': 'purple',
                 'TimesNet': 'darkred', 'N-HiTS': 'teal', 'Prophet': 'darkorange'}

for i, (mname, preds) in enumerate(forecasts_all.items()):
    fig.add_trace(go.Scatter(
        x=dates_test, y=preds, mode='lines',
        name=mname,
        line=dict(color=model_palette.get(mname, palette[i]), dash=line_styles[i % len(line_styles)], width=1.8)
    ))

fig.update_layout(
    title='Prévisions de tous les modèles vs Réel (zone test)',
    xaxis_title='Date', yaxis_title='Coût (€)',
    hovermode='x unified', height=500
)
fig.show()

---
## Prévision J+30 — Meilleur modèle

On ré-entraîne le meilleur modèle sur **toutes** les données disponibles (train + test) pour prédire les 30 prochains jours.

In [34]:
print(f'  Meilleur modèle sélectionné : {best_model}')
full_sf  = daily.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

last_date  = daily['ds'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=HORIZON, freq='D')

if best_model in ('AutoARIMA', 'AutoETS', 'AutoTheta'):
    if best_model == 'AutoARIMA':
        model_final = StatsForecast(models=[AutoARIMA(season_length=7, approximation=True)], freq='D', n_jobs=-1, verbose=False)
    elif best_model == 'AutoETS':
        model_final = StatsForecast(models=[AutoETS(season_length=7)], freq='D', n_jobs=-1, verbose=False)
    else:
        model_final = StatsForecast(models=[AutoTheta(season_length=7)], freq='D', n_jobs=-1, verbose=False)
    model_final.fit(full_sf)
    fc_future = model_final.predict(h=HORIZON)
    y_future = fc_future[best_model].values

elif best_model == 'TimesNet':
    timesnet_final = TimesNet(h=HORIZON, input_size=INPUT_SIZE, encoder_layers=2,
                               d_model=32, d_ff=64, dropout=0.1, top_k=3, num_kernels=4,
                               max_steps=300, batch_size=8, random_seed=42, scaler_type='standard')
    nf_final = NeuralForecast(models=[timesnet_final], freq='D')
    nf_final.fit(full_sf)
    fc_future = nf_final.predict()
    y_future = fc_future['TimesNet'].values

elif best_model == 'N-HiTS':
    nhits_final = NHITS(h=HORIZON, input_size=2*HORIZON, stack_types=['identity','identity','identity'],
                         n_blocks=[1,1,1], mlp_units=[[256,256],[256,256],[256,256]],
                         interpolation_mode='linear', dropout_prob_theta=0.1,
                         max_steps=300, batch_size=8, random_seed=42, scaler_type='standard')
    nf_final = NeuralForecast(models=[nhits_final], freq='D')
    nf_final.fit(full_sf)
    fc_future = nf_final.predict()
    y_future = fc_future['NHITS'].values

elif best_model == 'Prophet':
    m_final = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False,
                      changepoint_prior_scale=0.1, seasonality_prior_scale=5)
    m_final.fit(daily.rename(columns={'ds': 'ds', 'y': 'y'}))
    future_df = m_final.make_future_dataframe(periods=HORIZON)
    fc_future_prophet = m_final.predict(future_df)
    y_future = fc_future_prophet['yhat'].iloc[-HORIZON:].values

print(f'  Prévision sur {HORIZON} jours : {future_dates[0].date()} → {future_dates[-1].date()}')
print(f'  Total prévu  : {y_future.sum():,.2f} €')
print(f'  Moyenne/jour : {y_future.mean():,.2f} €/j')

fig = go.Figure()
context_all = daily.iloc[-60:]
fig.add_trace(go.Scatter(x=context_all['ds'], y=context_all['y'], mode='lines',
                         name='Historique', line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=future_dates, y=y_future, mode='lines+markers',
                         name=f'Prévision {best_model}',
                         line=dict(color='crimson', dash='dash', width=2),
                         marker=dict(size=5)))
fig.add_vrect(x0=future_dates[0], x1=future_dates[-1],
              fillcolor='rgba(255,0,0,0.05)', line_width=0,
              annotation_text='Zone prévision', annotation_position='top left')
fig.update_layout(
    title=f'Prévision J+{HORIZON} — {best_model} | Total estimé : {y_future.sum():,.0f} €',
    xaxis_title='Date', yaxis_title='Coût (€)', height=420
)
fig.show()

  Meilleur modèle sélectionné : AutoETS
  Prévision sur 30 jours : 2026-06-24 → 2026-07-23
  Total prévu  : 676.88 €
  Moyenne/jour : 22.56 €/j


---
## Prévision par service — Top 5

On applique le meilleur modèle (ou AutoARIMA par défaut pour sa robustesse) sur les 5 services les plus coûteux.

In [35]:
service_cols = [c for c in per_service.columns if c != 'ds']
top5_services = (
    per_service[service_cols].sum().sort_values(ascending=False).head(5).index.tolist()
)
print(f'  Top 5 services : {top5_services}')

service_forecasts = {}
n_svc_train = len(per_service) - HORIZON

for svc in top5_services:
    svc_series = per_service[['ds', svc]].rename(columns={svc: 'y'})
    svc_train  = svc_series.iloc[:n_svc_train].assign(unique_id=svc)

    sf_svc = StatsForecast(
        models=[AutoARIMA(season_length=7, approximation=True)],
        freq='D', n_jobs=-1, verbose=False
    )
    sf_svc.fit(svc_train)
    fc_svc = sf_svc.predict(h=HORIZON)
    y_svc_pred = fc_svc['AutoARIMA'].values

    y_svc_true = svc_series.iloc[-HORIZON:]['y'].values
    m = compute_metrics(y_svc_true, y_svc_pred, svc)
    service_forecasts[svc] = {'pred': y_svc_pred, 'true': y_svc_true, 'metrics': m}
    print(f"  {svc[:30]:<32} MAE={m['MAE']:.2f}€  R²={m['R²']:.4f}")

# Chart multi-services
fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=[s[:35] for s in top5_services], vertical_spacing=0.04)
color_list = px.colors.qualitative.Bold
for i, svc in enumerate(top5_services, start=1):
    svc_full = per_service[['ds', svc]].rename(columns={svc: 'y'})
    train_svc_full = svc_full.iloc[:-HORIZON]
    fig.add_trace(go.Scatter(x=train_svc_full['ds'], y=train_svc_full['y'], mode='lines',
                             name='Train', line=dict(color='steelblue', width=1),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=svc_full.iloc[-HORIZON:]['ds'], y=service_forecasts[svc]['true'],
                             mode='lines', name='Réel', line=dict(color='orange', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=svc_full.iloc[-HORIZON:]['ds'], y=service_forecasts[svc]['pred'],
                             mode='lines', name='Prévision', line=dict(color=color_list[i-1], dash='dash'),
                             showlegend=(i==1)), row=i, col=1)

fig.update_layout(title='Prévision AutoARIMA par service — Top 5', height=1000, showlegend=True)
fig.show()

  Top 5 services : ['Cloud SQL', 'BigQuery', 'Claude Sonnet 4.6', 'Cloud Spanner', 'Cloud Run']
  Cloud SQL                        MAE=0.30€  R²=-0.0225
  BigQuery                         MAE=3.40€  R²=-0.0638
  Claude Sonnet 4.6                MAE=2.80€  R²=0.0881
  Cloud Spanner                    MAE=0.09€  R²=-0.0284
  Cloud Run                        MAE=0.04€  R²=-0.0301


---
## Résumé exécutif

In [36]:
print('═' * 70)
print('   RÉSUMÉ BENCHMARK — Coûts GCP Veolia')
print('═' * 70)
print(f'  Données       : {len(daily)} jours ({daily["ds"].min().date()} → {daily["ds"].max().date()})')
print(f'  Horizon test  : {HORIZON} jours')
print(f'  Coût réel moy.: {y_test.mean():.2f} €/jour  |  Total test : {y_test.sum():.2f} €')
print()
print('  Classement final :')
for i, row in bench.iterrows():
    medal = ['🥇', '🥈', '🥉', '  4.', '  5.', '  6.'][i]
    print(f"  {medal} {row['Model']:<14}  R²={row['R²']:+.4f}  MAE={row['MAE']:.1f}€  MAPE={row['MAPE (%)']:.1f}%")
print()
print(f'  Meilleur modèle   : {best_model}')
print(f'  Prévision J+{HORIZON} : {y_future.sum():,.0f} € total  |  {y_future.mean():.0f} €/jour en moyenne')
print('═' * 70)

══════════════════════════════════════════════════════════════════════
   RÉSUMÉ BENCHMARK — Coûts GCP Veolia
══════════════════════════════════════════════════════════════════════
  Données       : 170 jours (2026-01-05 → 2026-06-23)
  Horizon test  : 30 jours
  Coût réel moy.: 22.95 €/jour  |  Total test : 688.39 €

  Classement final :
  🥇 AutoETS         R²=+0.0495  MAE=5.7€  MAPE=23.9%
  🥈 AutoTheta       R²=+0.0849  MAE=5.7€  MAPE=24.8%
  🥉 Prophet         R²=+0.1523  MAE=5.8€  MAPE=26.9%
    4. TimesNet        R²=-0.1453  MAE=6.5€  MAPE=29.6%
    5. AutoARIMA       R²=-0.2478  MAE=6.9€  MAPE=27.8%
    6. N-HiTS          R²=-0.7304  MAE=8.1€  MAPE=35.3%

  Meilleur modèle   : AutoETS
  Prévision J+30 : 677 € total  |  23 €/jour en moyenne
══════════════════════════════════════════════════════════════════════


---
## Section B — Analyse des événements Cloud Run

### Pourquoi ces métriques sont des "events" pour la prédiction de coût ?

| Signal | Lien avec le coût | Interprétation |
|--------|-------------------|----------------|
| **CPU utilization** | Direct : Cloud Run facture en CPU×temps | Pic CPU → surconsommation immédiate |
| **Instance count** | Direct : chaque instance active est facturée | Scaling-up → coût instantané |
| **Request count** | Indirect : trafic → instanciation → CPU | Burst de requêtes → cascade coût |
| **Request latency** | Indirect : latency élevée = instance qui tourne plus longtemps | Slow requests → CPU idle facturé |

Les CSV ont une granularité **6h** (4 points/jour). On les agrège à la journée pour les aligner sur la série de coût daily.

In [ ]:

# ── Chargement & parsing des 4 CSV Cloud Run ──────────────────────────────────
import re

CSV_FILES = {
    'cpu':       'Cloud Run Revision - Container CPU Utilization.csv',
    'instances': 'Cloud Run Revision - Instance Count.csv',
    'requests':  'Cloud Run Revision - Request Count.csv',
    'latency':   'Cloud Run Revision - Request Latency.csv',
}

def load_cloudrun_csv(path):
    """Parse le format non-standard : 2 lignes de header, puis 'date_str,value'."""
    with open(path, encoding='utf-8') as f:
        lines = f.readlines()
    rows = []
    for line in lines[2:]:          # skip TimeSeries ID + project_id
        line = line.strip()
        if not line:
            continue
        # split sur la dernière virgule pour isoler la valeur numérique
        last_comma = line.rfind(',')
        date_str = line[:last_comma].strip()
        val_str  = line[last_comma + 1:].strip()
        # nettoyer le fuseau "GMT+0100 (heure normale d'Europe centrale)"
        date_clean = re.sub(r'GMT[+-]\d{4}\s*\(.*?\)', '', date_str).strip()
        try:
            ts = pd.to_datetime(date_clean, format='%a %b %d %Y %H:%M:%S')
        except Exception:
            ts = pd.to_datetime(date_clean, infer_datetime_format=True)
        rows.append({'ts': ts, 'value': float(val_str)})
    return pd.DataFrame(rows).sort_values('ts').reset_index(drop=True)

cr_raw = {}
for key, fname in CSV_FILES.items():
    cr_raw[key] = load_cloudrun_csv(fname)
    df_ = cr_raw[key]
    print(f"  {key:<12} : {len(df_)} points  "
          f"{df_['ts'].min().date()} → {df_['ts'].max().date()}  "
          f"min={df_['value'].min():.3f}  max={df_['value'].max():.3f}")


In [ ]:

# ── Agrégation journalière + normalisation ────────────────────────────────────
cr_daily = {}
for key, df_ in cr_raw.items():
    agg = df_.copy()
    agg['date'] = agg['ts'].dt.date.astype(str)
    # CPU : somme (CPU-secondes accumulées) ; les autres : moyenne journalière
    func = 'sum' if key == 'cpu' else 'mean'
    cr_daily[key] = agg.groupby('date')['value'].agg(func).rename(key)

# Fusionner sur l'index de daily (ds en string)
events_df = pd.DataFrame({'date': daily['ds'].dt.date.astype(str)})
for key, series in cr_daily.items():
    events_df = events_df.merge(series.reset_index().rename(columns={'date': 'date'}),
                                on='date', how='left')

# Remplir les jours sans données Cloud Run par 0 (service inactif = coût nul)
events_df = events_df.fillna(0)

# Min-max normalisation par feature pour éviter l'effet d'échelle
from sklearn.preprocessing import MinMaxScaler

FEATURE_COLS = ['cpu', 'instances', 'requests', 'latency']
scaler_ev = MinMaxScaler()
events_df[FEATURE_COLS] = scaler_ev.fit_transform(events_df[FEATURE_COLS])

events_df['ds'] = daily['ds'].values
events_df = events_df.set_index('date')

print(f"  events_df : {events_df.shape}  —  couverture Cloud Run :")
for col in FEATURE_COLS:
    non_zero = (events_df[col] > 0).sum()
    print(f"    {col:<12} : {non_zero}/{len(events_df)} jours avec données ({non_zero/len(events_df)*100:.0f}%)")

events_df[FEATURE_COLS].head(5)


In [ ]:

# ── Exploration visuelle — signaux Cloud Run vs coût journalier ───────────────
fig = make_subplots(
    rows=5, cols=1, shared_xaxes=True,
    subplot_titles=['Coût journalier total (€)',
                    'CPU Utilization (normalisée)',
                    'Instance Count (normalisée)',
                    'Request Count (normalisée)',
                    'Request Latency (normalisée)'],
    vertical_spacing=0.04
)

series_plot = [
    (daily['ds'], daily['y'],        'steelblue', 'Coût'),
    (events_df['ds'], events_df['cpu'],       'crimson',  'CPU'),
    (events_df['ds'], events_df['instances'], 'darkorange','Instances'),
    (events_df['ds'], events_df['requests'],  'green',     'Requests'),
    (events_df['ds'], events_df['latency'],   'purple',    'Latency'),
]
for i, (x, y, color, name) in enumerate(series_plot, start=1):
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name=name,
                             line=dict(color=color, width=1.5)), row=i, col=1)

fig.update_layout(title='Signaux Cloud Run vs Coût GCP — Alignement temporel',
                  height=900, showlegend=False)
fig.show()


In [ ]:

# ── Analyse de corrélation croisée & lag optimaux ─────────────────────────────
from scipy import stats as sp_stats

cost_series = daily['y'].values

print('Corrélations Pearson (contemporaines) :\n')
corr_rows = []
for col in FEATURE_COLS:
    feat = events_df[col].values
    r, p = sp_stats.pearsonr(feat, cost_series)
    corr_rows.append({'Feature': col, 'r': round(r, 4), 'p': round(p, 4),
                      'Significatif': 'oui' if p < 0.05 else 'non'})
    print(f"  {col:<12}  r = {r:+.4f}  p = {p:.4f}  {'✓' if p < 0.05 else '—'}")

# Cross-corrélation avec lags 0..14 pour détecter les effets décalés
print('\nCross-corrélation avec lags (0→14 jours) :')
lag_results = {}
for col in FEATURE_COLS:
    feat = events_df[col].values
    lags, rs = [], []
    for lag in range(15):
        if lag == 0:
            r, _ = sp_stats.pearsonr(feat, cost_series)
        else:
            r, _ = sp_stats.pearsonr(feat[:-lag], cost_series[lag:])
        lags.append(lag)
        rs.append(r)
    best_lag = lags[np.argmax(np.abs(rs))]
    best_r   = rs[np.argmax(np.abs(rs))]
    lag_results[col] = {'lags': lags, 'rs': rs, 'best_lag': best_lag, 'best_r': best_r}
    print(f"  {col:<12}  meilleur lag = {best_lag}j   r_max = {best_r:+.4f}")

# Heatmap de corrélation features × coût
import plotly.figure_factory as ff

corr_data = events_df[FEATURE_COLS].copy()
corr_data['cost'] = cost_series
corr_matrix = corr_data.corr()

fig = px.imshow(corr_matrix, text_auto='.3f', color_continuous_scale='RdBu_r',
                color_continuous_midpoint=0,
                title='Matrice de corrélation — Features Cloud Run × Coût',
                height=420, aspect='auto')
fig.show()

# Cross-corrélation plot
fig2 = go.Figure()
colors_lag = {'cpu': 'crimson', 'instances': 'darkorange', 'requests': 'green', 'latency': 'purple'}
for col in FEATURE_COLS:
    fig2.add_trace(go.Scatter(x=lag_results[col]['lags'], y=lag_results[col]['rs'],
                              mode='lines+markers', name=col,
                              line=dict(color=colors_lag[col], width=2)))
fig2.add_hline(y=0, line_color='black', line_width=1)
fig2.add_hrect(y0=-1.96/np.sqrt(len(cost_series)), y1=1.96/np.sqrt(len(cost_series)),
               fillcolor='rgba(128,128,128,0.12)', line_width=0, annotation_text='IC 95%')
fig2.update_layout(title='Cross-corrélation Feature → Coût pour lags 0–14 jours',
                   xaxis_title='Lag (jours)', yaxis_title='Pearson r', height=380)
fig2.show()


In [ ]:

# ── Détection d'anomalies dans les signaux Cloud Run ─────────────────────────
# Un pic anormal dans CPU/instances/requests = "event" qui peut précéder un pic de coût

anomaly_events = {}
fig = make_subplots(rows=len(FEATURE_COLS), cols=1, shared_xaxes=True,
                    subplot_titles=[f'{col} — anomalies Z>2' for col in FEATURE_COLS],
                    vertical_spacing=0.06)

for i, col in enumerate(FEATURE_COLS, start=1):
    series = events_df[col].values
    mu, sigma = series.mean(), series.std()
    z = np.abs((series - mu) / (sigma + 1e-9))
    anomaly_mask = z > 2.0
    anomaly_dates = events_df['ds'][anomaly_mask].values
    anomaly_vals  = series[anomaly_mask]
    anomaly_events[col] = anomaly_dates

    fig.add_trace(go.Scatter(x=events_df['ds'], y=series, mode='lines',
                             name=col, line=dict(width=1.2)), row=i, col=1)
    if anomaly_mask.any():
        fig.add_trace(go.Scatter(x=anomaly_dates, y=anomaly_vals, mode='markers',
                                 name=f'{col} anomalie',
                                 marker=dict(color='red', size=8, symbol='x'),
                                 showlegend=False), row=i, col=1)
    fig.add_hline(y=mu + 2*sigma, line_color='orange', line_dash='dash',
                  line_width=1, row=i, col=1)

fig.update_layout(title='Événements anormaux dans les métriques Cloud Run (Z-score > 2)',
                  height=800, showlegend=False)
fig.show()

# Résumé des events détectés
print('\nRécapitulatif des événements détectés :')
for col, dates in anomaly_events.items():
    if len(dates):
        print(f"  {col:<12} : {len(dates)} événements")
        for d in dates[:5]:
            print(f"               → {pd.Timestamp(d).date()}")
        if len(dates) > 5:
            print(f"               ... ({len(dates)-5} de plus)")
    else:
        print(f"  {col:<12} : aucun événement")


In [ ]:

# ── Analyse spectrale FFT — fréquences dominantes des signaux Cloud Run ───────
# Même logique que les TimesBlock de TimesNet : FFT pour trouver les périodes dominantes

fig = make_subplots(rows=2, cols=3,
                    subplot_titles=['FFT — Coût total',
                                    'FFT — CPU', 'FFT — Instances',
                                    'FFT — Requests', 'FFT — Latency',
                                    'Périodes dominantes (jours)'])

all_signals = {'cost': daily['y'].values}
all_signals.update({col: events_df[col].values for col in FEATURE_COLS})
signal_colors = {'cost': 'steelblue', 'cpu': 'crimson', 'instances': 'darkorange',
                 'requests': 'green', 'latency': 'purple'}
positions = [(1,1), (1,2), (1,3), (2,1), (2,2)]

dominant_periods = {}
for idx, (name, sig) in enumerate(all_signals.items()):
    n = len(sig)
    fft_vals  = np.abs(np.fft.rfft(sig - sig.mean()))
    freqs     = np.fft.rfftfreq(n, d=1.0)   # unité = jours
    periods   = np.where(freqs > 0, 1.0 / freqs, np.inf)

    # Top 5 périodes (excluant infini et sub-journalière)
    valid = (periods < n) & (periods > 1.5)
    top_idx = np.argsort(fft_vals[valid])[-5:][::-1]
    valid_periods = periods[valid]
    valid_fft     = fft_vals[valid]
    top_periods   = valid_periods[top_idx]
    top_powers    = valid_fft[top_idx]
    dominant_periods[name] = list(zip(np.round(top_periods, 1), np.round(top_powers, 2)))

    if idx < 5:
        r, c = positions[idx]
        display_mask = (periods > 1.5) & (periods < n)
        fig.add_trace(go.Scatter(x=periods[display_mask], y=fft_vals[display_mask],
                                 mode='lines', name=name,
                                 line=dict(color=signal_colors[name], width=1.2)),
                      row=r, col=c)
        # Marquer les tops
        fig.add_trace(go.Scatter(x=top_periods, y=top_powers, mode='markers',
                                 marker=dict(color='red', size=7, symbol='diamond'),
                                 showlegend=False), row=r, col=c)

# Tableau des périodes dominantes
print('═' * 55)
print('  Périodes dominantes (FFT) par signal')
print('═' * 55)
for name, periods_list in dominant_periods.items():
    periods_str = '  |  '.join([f'{p:.1f}j' for p, _ in periods_list])
    print(f"  {name:<12} : {periods_str}")
print('═' * 55)

fig.update_xaxes(title_text='Période (jours)', type='log')
fig.update_layout(title='Analyse FFT — Fréquences dominantes de chaque signal (axe log)',
                  height=580, showlegend=False)
fig.show()


---
## Section C — Modèles Event-Aware

On ré-entraîne TimesNet et Prophet en leur injectant les 4 métriques Cloud Run comme covariables. L'objectif est de mesurer le **lift** obtenu par rapport aux versions baseline sans signal applicatif.

---
## Modèle 6 — TimesNet + Covariables Cloud Run *(Event-Aware)*

On ré-entraîne TimesNet en lui passant les 4 métriques Cloud Run comme **covariables historiques** (`hist_exog_list`). Le modèle peut ainsi apprendre comment les pics de CPU ou de latency précèdent les hausses de coût, et pondérer ses TimesBlock FFT en conséquence.

In [ ]:

# ── Modèle 6 : TimesNet + covariables Cloud Run ───────────────────────────────
# hist_exog_list = covariables passées uniquement sur la fenêtre historique (pas besoin de les projeter)

train_nf_ev = train.copy().assign(unique_id='total_cost')
for col in FEATURE_COLS:
    train_nf_ev[col] = events_df.loc[
        events_df['ds'].isin(train['ds']).values, col
    ].values

# top_k = périodes FFT dominantes détectées ci-dessus (on en garde 3)
TOP_K_FFT = 3

timesnet_ev = TimesNet(
    h=HORIZON,
    input_size=INPUT_SIZE,
    hist_exog_list=FEATURE_COLS,
    encoder_layers=2,
    d_model=32,
    d_ff=64,
    dropout=0.1,
    top_k=TOP_K_FFT,
    num_kernels=4,
    max_steps=400,
    batch_size=8,
    learning_rate=5e-4,
    random_seed=42,
    scaler_type='standard',
)

nf_timesnet_ev = NeuralForecast(models=[timesnet_ev], freq='D')
nf_timesnet_ev.fit(train_nf_ev)

# Pour predict(), on passe les covariables de la fenêtre test dans futr_df
# (hist_exog ne nécessite pas de valeurs futures — NeuralForecast les ignore au-delà du train)
fc_timesnet_ev = nf_timesnet_ev.predict()

y_pred_timesnet_ev = fc_timesnet_ev['TimesNet'].values
metrics_tn_ev = compute_metrics(y_test, y_pred_timesnet_ev, 'TimesNet+Events')
results.append(metrics_tn_ev)
forecasts_all['TimesNet+Events'] = y_pred_timesnet_ev

print(f"  TimesNet           — MAE={metrics_timesnet['MAE']:.2f}€  R²={metrics_timesnet['R²']:.4f}  (baseline)")
print(f"  TimesNet+Events    — MAE={metrics_tn_ev['MAE']:.2f}€  R²={metrics_tn_ev['R²']:.4f}  ← event-aware")
lift_mae = (metrics_timesnet['MAE'] - metrics_tn_ev['MAE']) / metrics_timesnet['MAE'] * 100
print(f"  Lift MAE           : {lift_mae:+.1f}%  ({'amélioration' if lift_mae > 0 else 'dégradation'})")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines',
                         name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines',
                         name='Réel', line=dict(color='orange', width=2)))
fig.add_trace(go.Scatter(x=test['ds'], y=y_pred_timesnet, mode='lines',
                         name='TimesNet (baseline)', line=dict(color='gray', dash='dot')))
fig.add_trace(go.Scatter(x=test['ds'], y=y_pred_timesnet_ev, mode='lines',
                         name='TimesNet+Events', line=dict(color='darkred', dash='dash', width=2)))
fig.update_layout(title=f"TimesNet+Events | R²={metrics_tn_ev['R²']:.4f} | MAE={metrics_tn_ev['MAE']:.1f}€ | Lift MAE {lift_mae:+.1f}%",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=400)
fig.show()


---
## Modèle 7 — Prophet + Covariables Cloud Run *(Event-Aware)*

Prophet accepte des **régresseurs externes** via `add_regressor()`. On injecte les 4 métriques Cloud Run (CPU, instances, requests, latency) agrégées à la journée comme covariables. Cela permet à Prophet d'apprendre la corrélation charge applicative → coût Cloud Run.

In [ ]:

# ── Modèle 7 : Prophet + régresseurs Cloud Run ────────────────────────────────
# Prophet injecte les features via add_regressor() — elles doivent être fournies
# aussi dans le future_df (on les projette en forward-fill depuis le dernier jour connu)

train_prophet_ev = train.copy().rename(columns={'ds': 'ds', 'y': 'y'})
for col in FEATURE_COLS:
    train_prophet_ev[col] = events_df.loc[
        events_df['ds'].isin(train['ds']).values, col
    ].values

prophet_ev = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.1,
    seasonality_prior_scale=5,
)
for col in FEATURE_COLS:
    prophet_ev.add_regressor(col, standardize=True)

prophet_ev.fit(train_prophet_ev)

# future_df avec projection des covariables : forward-fill sur les HORIZON jours
future_ev = prophet_ev.make_future_dataframe(periods=HORIZON)
# Aligner les features sur la période train (connues) + test (forward-fill)
full_features = events_df[['ds'] + FEATURE_COLS].copy()
future_ev = future_ev.merge(full_features, on='ds', how='left')
# Forward-fill pour les jours au-delà des données Cloud Run
future_ev[FEATURE_COLS] = future_ev[FEATURE_COLS].ffill().bfill().fillna(0)

fc_prophet_ev = prophet_ev.predict(future_ev)
y_pred_prophet_ev = fc_prophet_ev['yhat'].iloc[-HORIZON:].values

metrics_p_ev = compute_metrics(y_test, y_pred_prophet_ev, 'Prophet+Events')
results.append(metrics_p_ev)
forecasts_all['Prophet+Events'] = y_pred_prophet_ev

print(f"  Prophet            — MAE={metrics_prophet['MAE']:.2f}€  R²={metrics_prophet['R²']:.4f}  (baseline)")
print(f"  Prophet+Events     — MAE={metrics_p_ev['MAE']:.2f}€  R²={metrics_p_ev['R²']:.4f}  ← event-aware")
lift_p = (metrics_prophet['MAE'] - metrics_p_ev['MAE']) / metrics_prophet['MAE'] * 100
print(f"  Lift MAE           : {lift_p:+.1f}%  ({'amélioration' if lift_p > 0 else 'dégradation'})")

prophet_test_ev = fc_prophet_ev.iloc[-HORIZON:]
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines',
                         name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines',
                         name='Réel', line=dict(color='orange', width=2)))
fig.add_trace(go.Scatter(
    x=pd.concat([prophet_test_ev['ds'], prophet_test_ev['ds'].iloc[::-1]]),
    y=pd.concat([prophet_test_ev['yhat_upper'], prophet_test_ev['yhat_lower'].iloc[::-1]]),
    fill='toself', fillcolor='rgba(220,50,50,0.10)',
    line=dict(color='rgba(255,255,255,0)'), name='IC 80%'))
fig.add_trace(go.Scatter(x=test['ds'], y=y_pred_prophet, mode='lines',
                         name='Prophet (baseline)', line=dict(color='gray', dash='dot')))
fig.add_trace(go.Scatter(x=prophet_test_ev['ds'], y=prophet_test_ev['yhat'], mode='lines',
                         name='Prophet+Events', line=dict(color='crimson', dash='dash', width=2)))
fig.update_layout(title=f"Prophet+Events | R²={metrics_p_ev['R²']:.4f} | MAE={metrics_p_ev['MAE']:.1f}€ | Lift MAE {lift_p:+.1f}%",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=400)
fig.show()

# Importance des régresseurs Prophet (via les composantes)
component_cols = [c for c in fc_prophet_ev.columns if c in FEATURE_COLS]
if component_cols:
    print('\n  Contributions moyennes des régresseurs sur la période test :')
    for col in component_cols:
        contrib = fc_prophet_ev[col].iloc[-HORIZON:].mean()
        print(f"    {col:<12} : {contrib:+.4f} €/j en moyenne")


---
## Résumé exécutif — Baseline + Event-Aware

In [ ]:

# ── Tableau final Baseline vs Event-Aware ─────────────────────────────────────
all_metrics = results.copy()
for name, pred in [
    ('TimesNet+Events', forecasts_all.get('TimesNet+Events')),
    ('Prophet+Events',  forecasts_all.get('Prophet+Events')),
]:
    if pred is not None:
        all_metrics.append(compute_metrics(y_test, pred, name))

bench_final = pd.DataFrame(all_metrics)
bench_final['Rank_MAE']  = bench_final['MAE'].rank()
bench_final['Rank_RMSE'] = bench_final['RMSE'].rank()
bench_final['Rank_MAPE'] = bench_final['MAPE (%)'].rank()
bench_final['Rank_R2']   = bench_final['R²'].rank(ascending=False)
bench_final['Score']     = (bench_final[['Rank_MAE','Rank_RMSE','Rank_MAPE','Rank_R2']].mean(axis=1))
bench_final = bench_final.sort_values('Score').reset_index(drop=True)
bench_final['Type'] = bench_final['Model'].apply(lambda m: '★ Event-Aware' if '+Events' in m else 'Baseline')

print('═' * 82)
print('   BENCHMARK FINAL — Baseline vs Event-Aware (Score = rang moyen ↓)')
print('═' * 82)
print(f"  {'#':<3} {'Modèle':<22} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8} {'Score':>7}  Type")
print('-' * 82)
for i, row in bench_final.iterrows():
    tag = ' ◄' if i == 0 else ''
    print(f"  {i+1:<3} {row['Model']:<22} {row['MAE']:>7.2f}€ {row['RMSE']:>7.2f}€ "
          f"{row['MAPE (%)']:>7.1f}% {row['R²']:>8.4f} {row['Score']:>7.2f}  {row['Type']}{tag}")
print('═' * 82)

# ── Graphique métriques ────────────────────────────────────────────────────────
fig_cmp = make_subplots(rows=1, cols=2, subplot_titles=['MAE (€) — moins = mieux', 'R² — plus = mieux'])
ev_colors = ['crimson' if '+Events' in m else 'steelblue' for m in bench_final['Model']]
fig_cmp.add_trace(
    go.Bar(x=bench_final['Model'], y=bench_final['MAE'], marker_color=ev_colors,
           text=[f"{v:.1f}€" for v in bench_final['MAE']], textposition='outside'),
    row=1, col=1)
fig_cmp.add_trace(
    go.Bar(x=bench_final['Model'], y=bench_final['R²'], marker_color=ev_colors,
           text=[f"{v:.4f}" for v in bench_final['R²']], textposition='outside'),
    row=1, col=2)
fig_cmp.update_layout(title='Baseline (bleu) vs Event-Aware (rouge)', showlegend=False, height=450)
fig_cmp.update_xaxes(tickangle=-35)
fig_cmp.show()

# ── Lift MAE ───────────────────────────────────────────────────────────────────
baseline_mae_map = {r['Model']: r['MAE'] for _, r in bench_final.iterrows() if '+Events' not in r['Model']}
lift_rows = []
for _, row in bench_final.iterrows():
    if '+Events' in row['Model']:
        base = row['Model'].replace('+Events', '')
        if base in baseline_mae_map:
            lift = (baseline_mae_map[base] - row['MAE']) / (baseline_mae_map[base] + 1e-9) * 100
            lift_rows.append({'Modèle': row['Model'], 'MAE Lift (%)': round(lift, 2)})
if lift_rows:
    lift_df = pd.DataFrame(lift_rows)
    fig_lift = px.bar(lift_df, x='Modèle', y='MAE Lift (%)',
                      title='Réduction de l\'erreur MAE grâce aux covariables Cloud Run (+ = mieux)',
                      color='MAE Lift (%)', color_continuous_scale='RdYlGn', color_continuous_midpoint=0,
                      text=[f"{v:+.1f}%" for v in lift_df['MAE Lift (%)']], height=350)
    fig_lift.add_hline(y=0, line_color='black', line_width=1)
    fig_lift.update_traces(textposition='outside')
    fig_lift.update_coloraxes(showscale=False)
    fig_lift.show()

# ── Overlay toutes les prévisions ─────────────────────────────────────────────
pal = px.colors.qualitative.Bold
fig_all = go.Figure()
fig_all.add_trace(go.Scatter(x=train.iloc[-20:]['ds'], y=train.iloc[-20:]['y'], mode='lines',
                              name='Contexte', line=dict(color='lightgray', width=1.5)))
fig_all.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines+markers',
                              name='Réel', line=dict(color='black', width=2.5), marker=dict(size=5)))
for i, (mname, preds) in enumerate(forecasts_all.items()):
    fig_all.add_trace(go.Scatter(
        x=dates_test, y=preds, mode='lines', name=mname,
        line=dict(color=pal[i % len(pal)],
                  dash='dash' if '+Events' in mname else 'dot',
                  width=2.2 if '+Events' in mname else 1.6)))
fig_all.update_layout(title='Toutes prévisions — tiret=Event-Aware, pointillé=Baseline, noir=Réel',
                       xaxis_title='Date', yaxis_title='Coût (€)', hovermode='x unified', height=520)
fig_all.show()
